# Corpus Studies

_**melody-features**_ makes it easy to compare and contrast melodies with other melodies drawn from a corpus. In this notebook, we demonstrate a couple of common approaches to this task.

The first approach involves comparing feature values for a given melody to the range of values observed in the reference corpus. The second approach uses FANTASTIC's _m_-types to measure token prevalence with regards to a corpus.

## Included Corpora

We ship two corpora with the package, which are easy to import and use.

In [5]:
from melody_features import get_corpus_path
from melody_features.corpus import list_available_corpora, get_corpus_path

print(f"Available corpora: {list_available_corpora()}")
essen_path = get_corpus_path("essen")
print(essen_path)


Available corpora: ['essen', 'pearce_default_idyom']
/Users/davidwhyatt/feature_set/src/melody_features/corpora/essen_folksong_collection


# 1. Feature values across a corpus

We will start by assigning our target melody, which we will compare to the corpus. In this example, we will just use the first melody in the Essen Folksong Collection, `appenzel.mid`.

In [6]:
from melody_features.corpus import get_corpus_files
from pathlib import Path

# you could replace these lines with the path to your own midi file
# e.g. input_path = Path("path/to/your/file.mid")

input_list = get_corpus_files("essen", 1)
input_path = Path(input_list[0])
print(input_path)

/Users/davidwhyatt/feature_set/src/melody_features/corpora/essen_folksong_collection/appenzel.mid


Next, we need to compute features for our target melody, as well as for all the melodies to which we will compare it.

In [38]:
from melody_features import get_all_features

corpus_melodies = get_corpus_files("essen")
print(f"Found {len(corpus_melodies)} melodies in the corpus")

# compute features for the target melody and corpus all at once
melody_list = [input_path] + corpus_melodies
print(f"Computing features for {len(melody_list)} melodies")

df = get_all_features(input=melody_list, skip_idyom=True)

11:46:35 - melody_features - INFO - Starting feature extraction job...
11:46:35 - melody_features - INFO - Configuration Parameters:
11:46:35 - melody_features - INFO -   Key Estimation Strategy: infer_if_necessary
11:46:35 - melody_features - INFO -   Key Finding Algorithm: krumhansl_schmuckler
11:46:35 - melody_features - INFO -   Corpus Path: /Users/davidwhyatt/feature_set/src/melody_features/corpora/pearce_default_idyom
11:46:35 - melody_features - INFO -   IDyOM Configurations: 4 config(s)
11:46:35 - melody_features - INFO -     [pitch_stm]:
11:46:35 - melody_features - INFO -       Models: :stm
11:46:35 - melody_features - INFO -       Corpus: Using Corpus Path from Config
11:46:35 - melody_features - INFO -       Target Viewpoints: ['cpitch']
11:46:35 - melody_features - INFO -       Source Viewpoints: [('cpitch', 'cpint', 'cpintfref')]
11:46:35 - melody_features - INFO -       PPM Order: None
11:46:35 - melody_features - INFO -     [pitch_ltm]:
11:46:35 - melody_features - INFO

Found 8472 melodies in the corpus
Computing features for 8473 melodies


11:46:36 - melody_features - WARNING - Skipping polyphonic file: /Users/davidwhyatt/feature_set/src/melody_features/corpora/essen_folksong_collection/deut494.mid
11:46:36 - melody_features - WARNING - Skipping polyphonic file: /Users/davidwhyatt/feature_set/src/melody_features/corpora/essen_folksong_collection/deut0574.mid
11:46:37 - melody_features - WARNING - Skipping polyphonic file: /Users/davidwhyatt/feature_set/src/melody_features/corpora/essen_folksong_collection/deut1576.mid
11:46:38 - melody_features - WARNING - Skipping polyphonic file: /Users/davidwhyatt/feature_set/src/melody_features/corpora/essen_folksong_collection/deut2244.mid
11:46:39 - melody_features - WARNING - Skipping polyphonic file: /Users/davidwhyatt/feature_set/src/melody_features/corpora/essen_folksong_collection/deut2371.mid
11:46:39 - melody_features - WARNING - Skipping polyphonic file: /Users/davidwhyatt/feature_set/src/melody_features/corpora/essen_folksong_collection/deut2570.mid
11:46:39 - melody_featu

In [ ]:
# clean up data: keep 'melody_id' and all numeric columns, drop zero-variance cols, deduplicate
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cols = ['melody_id'] + [col for col in numeric_cols if col != 'melody_id']

df_subset = df[cols].copy()
# Remove zero variance columns (excluding 'melody_id')
zero_var_cols = [col for col in df_subset.columns if col != 'melody_id' and df_subset[col].nunique() <= 1]
df_subset = df_subset.drop(columns=zero_var_cols)

df_clean = df_subset.drop_duplicates()

Now that we have all the features in a dataframe, let's pick a simple one and plot the target melody against the distribution in the corpus.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

target_feature_widget = widgets.Dropdown(options=df_clean.columns.tolist(), description="Feature:")
display(target_feature_widget)
confirm_button = widgets.Button(description="Select Feature")
output = widgets.Output()

display(confirm_button, output)
selected_feature = {}

def on_confirm_clicked(b):
    with output:
        output.clear_output()
        print(f"Selected feature: {target_feature_widget.value}")
    selected_feature["feature"] = target_feature_widget.value

    global target_feature
    target_feature = target_feature_widget.value

confirm_button.on_click(on_confirm_clicked)

Dropdown(description='Feature:', options=('melody_id', 'melody_num', 'absolute_pitch.first_pitch', 'absolute_p…

Button(description='Select Feature', style=ButtonStyle())

Output()

In [52]:
# run this cell to enable plotting - then the button will automatically plot the feature distribution!
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
from scipy.stats import gaussian_kde
from matplotlib.collections import LineCollection

def plot_feature_distribution(feature_name):
    data = df_clean[feature_name].dropna()
    plt.figure(figsize=(10, 6))

    n, bins, patches = plt.hist(data, bins=30, alpha=0.8, label="Corpus", edgecolor='none')

    norm = mpl.colors.Normalize(vmin=bins[0], vmax=bins[-1])
    cmap = plt.get_cmap('viridis')

    for bin_left, bin_right, patch in zip(bins[:-1], bins[1:], patches):
        color_val = (bin_left + bin_right) / 2
        patch.set_facecolor(cmap(norm(color_val)))

    kde = gaussian_kde(data)
    x_points = np.linspace(min(data), max(data), 1000)
    y_points = kde(x_points)
    area_hist = np.sum(n) * (bins[1] - bins[0])
    scaled_y = y_points * area_hist

    segments = np.array([x_points, scaled_y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([segments[:-1], segments[1:]], axis=1)

    lc_colors = cmap(norm((x_points[:-1] + x_points[1:]) / 2))

    lc = LineCollection(segments, colors=lc_colors, linewidth=2, label="Density")
    plt.gca().add_collection(lc)

    target_row = df_clean[df_clean['melody_id'] == str(input_path)]
    if not target_row.empty:
        target_val = target_row[feature_name].iloc[0]
        plt.axvline(target_val, color="red", label="Target", linewidth=2)
    else:
        print("Target melody not found in DataFrame.")

    plt.xlabel(feature_name.replace(".", " ").title())
    plt.ylabel("Melody Count")
    plt.legend()
    plt.tight_layout()
    plt.show()

# automatically call plot_feature_distribution when the feature is confirmed
def on_confirm_clicked(b):
    with output:
        output.clear_output()
        print(f"Selected feature: {target_feature_widget.value}")
        selected_feature["feature"] = target_feature_widget.value
        global target_feature
        target_feature = target_feature_widget.value
        plot_feature_distribution(target_feature)

confirm_button.on_click(on_confirm_clicked)


This makes it very easy to see how our chosen melody compares to the corpus! Why not try changing `target_feature` above, and see how your chosen melody compares to the corpus acros a range of different features? You might find it useful to refer to the [documentation](https://dmwhyatt.github.io/melody-features/api/feature_definitions/).

## 2. FANTASTIC Corpus Analyses

The package includes several features that can be used to explicitly compare melodies to a representative corpus. These features can help answer interesting research questions, such as "how similar is this melody to those drawn from a certain musical tradition?"

The package's `corpus` module facilitates one such approach, using [FANTASTIC's _m_-types](https://www.doc.gold.ac.uk/isms/m4s/FANTASTIC_docs.pdf) to characterise melodies within a corpus. 

In FANTASTIC, melodies within a corpus are broken down into small melodic units called _m_-types. An m-type is, effectively, a musical _n_-gram, a small chunk of a larger phrase. By looking at all of the different 1-note, 2-note etc. chunks a melody contains, we may be able to identify patterns and syntax that describes how melodies are constructed. 

This is especially useful over a corpus, where we can collect lots of data about these melodic patterns. A simple approach might involve iterating over every melody in the corpus, counting the appearance of each _m_-type, and aggregating counts of the same _m_-type together. We can do this easily in _**melody-features**_ using the `make_corpus_stats()` function on the desired corpus directory.

In [10]:
from melody_features.corpus import make_corpus_stats, get_corpus_path

# name of file to which we will save the corpus stats
essen_corpus = get_corpus_path("essen")
corpus_stats = "essen_corpus_stats.json"
make_corpus_stats(midi_dir=essen_corpus, output_file=corpus_stats)
print(f"Saved corpus stats to {corpus_stats}")

10:38:33 - melody_features - INFO - Found 8472 MIDI files
Loading MIDI files: 100%|██████████| 8472/8472 [00:03<00:00, 2399.42it/s]
10:38:37 - melody_features - INFO - Processing 8470 valid melodies
Computing n-grams: 100%|██████████| 8470/8470 [00:04<00:00, 1876.86it/s]
10:38:44 - melody_features - INFO - Corpus statistics saved and loaded successfully.
10:38:44 - melody_features - INFO - Corpus size: 8470 melodies
10:38:44 - melody_features - INFO - N-gram lengths: [1, 5]


Saved corpus stats to essen_corpus_stats.json


The generated `essen_corpus_stats.json` file contains the counts of all the _m_-types between lengths 1 and 5 (inclusive). We can clearly see that some melodic movements are very common, occuring thousands of times in the corpus, and some are very rare.

Though these counts may be useful on their own in some cases, they are generally more useful as a reference for corpus-based measures of prevalence. We can compute features that quantify how prevalent the _m_-types in our target melody are compared to the distribution of the _m_-types in the corpus.

In [11]:
from melody_features import get_corpus_features
from melody_features.corpus import load_corpus_stats
from melody_features.io import load_midi
import pandas as pd

# we load the corpus stats into a dict, and read our midi file in (here using the same input melody as before)
corpus_dict = load_corpus_stats(corpus_stats)
mel = load_midi(input_path)

# now compute the features
corpus_features = get_corpus_features(mel, corpus_dict, phrase_gap=0.5, max_ngram_order=5)

# and convert the features into a dataframe
df_corpus = pd.DataFrame(data=[corpus_features], index=["Input Melody"])
df_corpus.head()

,tfdf_spearman,tfdf_kendall,mean_log_tfdf,norm_log_dist,max_log_df,min_log_df,mean_log_df,mean_global_local_weight,std_global_local_weight,mean_global_weight,std_global_weight
Input Melody,0.317586,0.259944,0.000012,0.003061,12.982281,0.0,4.48301,1.26423,0.511888,1.001282,0.03838


These features describe some interesting properties of our input melody. For example, the value for `max_log_df` corresponds to log<sub>2</sub> of the count of the _m_-type most prevalent in the corpus. Here, our target melody contained an _m_-type that was contained in 8092 melodies in the corpus! This tells us that at least one part of the melody is very common in the corpus, and this might increase how likely we are to consider that part of the melody as 'familiar'.

Perhaps a more holistic measure is the `mean_log_df`, which takes the average of the log<sub>2</sub> counts from the corpus across all _m_-types that appear in the melody. As a result, it ought to capture some aspect of how common on average the melody is with regards to that corpus. High values for this feature correspond to melodies that contain many melodic components that are well-represented in the corpus.